# Find Sentinel2/ECOSTRESS matches

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import rasterio as rio

from datetime import datetime, timedelta
from pyproj import CRS
from rasterio.warp import transform_bounds
from shapely.geometry import Polygon
from tqdm import tqdm

In [3]:
from sensorsio.mgrs import get_mgrs_tiles_from_roi

In [4]:
from etdataset.database import (
    create_s2_db,
    create_eco_db,
    to_collectionV2,
)
from etdataset.selection import select_products
from etdataset.utils import get_bbox_from_mgrs_tile, get_bbox_from_roi

### Inputs

To be modified 

In [5]:
# Databases paths
metadata_path = os.environ[
    "METADATA_PATH"
]  # If you have defined the environment variable
# metadata_path = to_be_modified
eco_db_path = os.path.join(metadata_path, "ecostress_eco2lste_655f877cf0476b91.csv")
s2_db_path = os.path.join(metadata_path, "SENTINEL2.csv")

In [6]:
# ROI
s2_tile_id = "28PCA"
data_path = os.environ["DATA_PATH"]
roi_path = os.path.join(data_path, "zones", "Zone_Senegal_Centre.shp")

In [7]:
# Acquisition dates
min_date_str = "2023-02-01"  # Format YYYY-MM-DD
max_date_str = "2023-03-31"  # Format YYYY-MM-DD

In [8]:
# Delta time between two acquisitions
delta_time_str = "3 days"

In [9]:
# Other criteria to filter products
min_roi_overlap = 33
min_product_overlap = 40

In [10]:
# Outpit directory
output_path = "out"
os.makedirs(output_path, exist_ok=True)

### Prepare

In [11]:
# Convert dates
min_date = datetime.strptime(min_date_str, "%Y-%m-%d")
max_date = datetime.strptime(max_date_str, "%Y-%m-%d")
delta = pd.Timedelta(delta_time_str)

In [12]:
# Get bounding box and CRS for a specific tile
s2_tile_bbox, s2_tile_crs = get_bbox_from_mgrs_tile(s2_tile_id)

In [13]:
# Get bounding box and CRS for a ROI
roi_bbox, roi_crs = get_bbox_from_roi(roi_path)

### To obtain a list of MGRS tiles that overlap a ROI with the percentage of surface area covered

In [14]:
mgrs_tiles = get_mgrs_tiles_from_roi(roi_bbox, roi_crs)

In [15]:
mgrs_tiles

,Name,overlap_geometry,geometry,overlap_percentage
0,28PBA,"POLYGON ((-16.76482 14.46621, -16.75975 13.800...","POLYGON ((-17.78282 14.45638, -16.76482 14.466...",22.271238
1,28PBB,"POLYGON ((-16.76412 14.37783, -17.10000 14.374...","POLYGON ((-17.79446 15.35961, -16.77221 15.370...",20.551890
2,28PCA,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120620
3,28PCB,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627789
4,28PDA,"POLYGON ((-15.90000 14.47105, -15.90000 13.800...","POLYGON ((-15.92826 14.47100, -14.90943 14.472...",1.789794
5,28PDB,"POLYGON ((-15.92790 14.38258, -15.93054 15.000...","POLYGON ((-15.93215 15.37519, -14.90905 15.377...",1.780280


### Find ECOSTRESS products

In [16]:
eco_db = create_eco_db(
    eco_db_path,
    min_date=min_date,
    max_date=max_date,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    min_roi_overlap=min_roi_overlap,
)

24-01-24 17:52:57 :: INFO :: Found 20 ECOSTRESS products fullfilling the requirements


In [17]:
eco_db_v2 = to_collectionV2(eco_db, roi_bbox, roi_crs, min_roi_overlap)

In [18]:
eco_db_v2

,product_name,date,overlap_geometry,overlap_percentage,geometry
0,ECOv002_L2T_LSTE_26128_016_28PCA_20230213T1738...,2023-02-13,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
1,ECOv002_L2T_LSTE_26128_016_28PCB_20230213T1738...,2023-02-13,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
2,ECOv002_L2T_LSTE_26372_015_28PCA_20230301T1115...,2023-03-01,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
3,ECOv002_L2T_LSTE_26372_015_28PCB_20230301T1115...,2023-03-01,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
4,ECOv002_L2T_LSTE_26372_016_28PCA_20230301T1116...,2023-03-01,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
5,ECOv002_L2T_LSTE_26372_016_28PCB_20230301T1116...,2023-03-01,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
6,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
7,ECOv002_L2T_LSTE_26381_003_28PCB_20230302T0034...,2023-03-02,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
8,ECOv002_L2T_LSTE_26433_014_28PCA_20230305T0940...,2023-03-05,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
9,ECOv002_L2T_LSTE_26433_014_28PCB_20230305T0940...,2023-03-05,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."


### Find Sentinel2 products

In [19]:
s2_db = create_s2_db(
    s2_db_path,
    min_date=min_date,
    max_date=max_date,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    min_roi_overlap=min_roi_overlap,
)

24-01-24 17:53:08 :: INFO :: Found 47 Sentinel2 products fullfilling the requirements


In [20]:
s2_db

,product_name,date,overlap_geometry,geometry,overlap_percentage
0,SENTINEL2B_20230201-114720-111_L2A_T28PCB,2023-02-01,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
1,SENTINEL2A_20230203-113728-531_L2A_T28PCB,2023-02-03,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
2,SENTINEL2A_20230206-114722-028_L2A_T28PCB,2023-02-06,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
3,SENTINEL2B_20230208-113728-255_L2A_T28PCB,2023-02-08,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
4,SENTINEL2B_20230211-114721-668_L2A_T28PCB,2023-02-11,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
5,SENTINEL2A_20230216-114719-598_L2A_T28PCB,2023-02-16,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
6,SENTINEL2B_20230218-113728-549_L2A_T28PCB,2023-02-18,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
7,SENTINEL2B_20230221-114722-722_L2A_T28PCB,2023-02-21,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
8,SENTINEL2A_20230223-113727-370_L2A_T28PCB,2023-02-23,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459
9,SENTINEL2A_20230226-114720-692_L2A_T28PCB,2023-02-26,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",41.234459


### Select matches

In [21]:
res = select_products(s2_db, eco_db_v2, delta, min_product_overlap)

Matching datasets ...: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:00<00:00, 26.85it/s]
24-01-24 17:53:09 :: INFO :: Found 111 matches


In [22]:
res

,product_name_1,date_1,product_name_2,date_2,overlap
0,SENTINEL2B_20230211-114735-878_L2A_T28PCA,2023-02-11,ECOv002_L2T_LSTE_26128_016_28PCA_20230213T1738...,2023-02-13,100.0
1,SENTINEL2B_20230211-114721-668_L2A_T28PCB,2023-02-11,ECOv002_L2T_LSTE_26128_016_28PCB_20230213T1738...,2023-02-13,100.0
2,SENTINEL2A_20230216-114733-797_L2A_T28PCA,2023-02-16,ECOv002_L2T_LSTE_26128_016_28PCA_20230213T1738...,2023-02-13,100.0
3,SENTINEL2A_20230216-114719-598_L2A_T28PCB,2023-02-16,ECOv002_L2T_LSTE_26128_016_28PCB_20230213T1738...,2023-02-13,100.0
4,SENTINEL2A_20230226-114734-900_L2A_T28PCA,2023-02-26,ECOv002_L2T_LSTE_26372_015_28PCA_20230301T1115...,2023-03-01,100.0
...,...,...,...,...,...
106,SENTINEL2B_20230330-113745-671_L2A_T28PCA,2023-03-30,ECOv002_L2T_LSTE_26800_017_28PCA_20230329T0015...,2023-03-29,100.0
107,SENTINEL2B_20230330-113733-106_L2A_T28PCB,2023-03-30,ECOv002_L2T_LSTE_26800_017_28PCB_20230329T0015...,2023-03-29,100.0
108,SENTINEL2B_20230330-113745-671_L2A_T28PCA,2023-03-30,ECOv002_L2T_LSTE_26809_001_28PCA_20230329T1333...,2023-03-29,100.0
109,SENTINEL2B_20230330-113733-106_L2A_T28PCB,2023-03-30,ECOv002_L2T_LSTE_26809_001_28PCB_20230329T1333...,2023-03-29,100.0


### List of products

In [23]:
list(res["product_name_1"].unique())

['SENTINEL2B_20230211-114735-878_L2A_T28PCA',
 'SENTINEL2B_20230211-114721-668_L2A_T28PCB',
 'SENTINEL2A_20230216-114733-797_L2A_T28PCA',
 'SENTINEL2A_20230216-114719-598_L2A_T28PCB',
 'SENTINEL2A_20230226-114734-900_L2A_T28PCA',
 'SENTINEL2A_20230226-114720-692_L2A_T28PCB',
 'SENTINEL2B_20230228-113743-121_L2A_T28PCA',
 'SENTINEL2B_20230228-113730-663_L2A_T28PCB',
 'SENTINEL2B_20230303-114738-815_L2A_T28PCA',
 'SENTINEL2B_20230303-114724-615_L2A_T28PCB',
 'SENTINEL2A_20230305-113738-949_L2A_T28PCA',
 'SENTINEL2A_20230305-113726-514_L2A_T28PCB',
 'SENTINEL2A_20230308-114732-624_L2A_T28PCA',
 'SENTINEL2A_20230308-114718-413_L2A_T28PCB',
 'SENTINEL2B_20230310-113746-352_L2A_T28PCA',
 'SENTINEL2B_20230310-113733-720_L2A_T28PCB',
 'SENTINEL2A_20230315-113739-530_L2A_T28PCA',
 'SENTINEL2A_20230315-113727-051_L2A_T28PCB',
 'SENTINEL2A_20230318-114735-293_L2A_T28PCA',
 'SENTINEL2A_20230318-114721-093_L2A_T28PCB',
 'SENTINEL2B_20230320-113745-436_L2A_T28PCA',
 'SENTINEL2B_20230320-113732-915_L

In [24]:
list(res["product_name_2"].unique())

['ECOv002_L2T_LSTE_26128_016_28PCA_20230213T173802_0601_01',
 'ECOv002_L2T_LSTE_26128_016_28PCB_20230213T173802_0601_01',
 'ECOv002_L2T_LSTE_26372_015_28PCA_20230301T111515_0601_02',
 'ECOv002_L2T_LSTE_26372_015_28PCB_20230301T111515_0601_02',
 'ECOv002_L2T_LSTE_26372_016_28PCA_20230301T111607_0601_02',
 'ECOv002_L2T_LSTE_26372_016_28PCB_20230301T111607_0601_02',
 'ECOv002_L2T_LSTE_26381_003_28PCA_20230302T003454_0601_02',
 'ECOv002_L2T_LSTE_26381_003_28PCB_20230302T003454_0601_02',
 'ECOv002_L2T_LSTE_26433_014_28PCA_20230305T094027_0601_01',
 'ECOv002_L2T_LSTE_26433_014_28PCB_20230305T094027_0601_01',
 'ECOv002_L2T_LSTE_26433_015_28PCA_20230305T094119_0601_01',
 'ECOv002_L2T_LSTE_26433_015_28PCB_20230305T094119_0601_01',
 'ECOv002_L2T_LSTE_26494_014_28PCA_20230309T080538_0601_01',
 'ECOv002_L2T_LSTE_26494_014_28PCB_20230309T080538_0601_01',
 'ECOv002_L2T_LSTE_26494_015_28PCA_20230309T080630_0601_01',
 'ECOv002_L2T_LSTE_26494_015_28PCB_20230309T080630_0601_01',
 'ECOv002_L2T_LSTE_26616

### List of pairs

In [25]:
for product_name_1, group in res.groupby("product_name_1"):
    print(
        f"Image: {product_name_1} - List of images:  {list(group['product_name_2'].unique())}\n"
    )

Image: SENTINEL2A_20230213-113740-074_L2A_T28PCA - List of images:  ['ECOv002_L2T_LSTE_26128_016_28PCA_20230213T173802_0601_01']

Image: SENTINEL2A_20230216-114719-598_L2A_T28PCB - List of images:  ['ECOv002_L2T_LSTE_26128_016_28PCB_20230213T173802_0601_01']

Image: SENTINEL2A_20230216-114733-797_L2A_T28PCA - List of images:  ['ECOv002_L2T_LSTE_26128_016_28PCA_20230213T173802_0601_01']

Image: SENTINEL2A_20230226-114720-692_L2A_T28PCB - List of images:  ['ECOv002_L2T_LSTE_26372_015_28PCB_20230301T111515_0601_02', 'ECOv002_L2T_LSTE_26372_016_28PCB_20230301T111607_0601_02']

Image: SENTINEL2A_20230226-114734-900_L2A_T28PCA - List of images:  ['ECOv002_L2T_LSTE_26372_015_28PCA_20230301T111515_0601_02', 'ECOv002_L2T_LSTE_26372_016_28PCA_20230301T111607_0601_02']

Image: SENTINEL2A_20230305-113726-514_L2A_T28PCB - List of images:  ['ECOv002_L2T_LSTE_26381_003_28PCB_20230302T003454_0601_02', 'ECOv002_L2T_LSTE_26433_014_28PCB_20230305T094027_0601_01', 'ECOv002_L2T_LSTE_26433_015_28PCB_2023030

### Write results

In [26]:
res.to_csv(os.path.join(output_path, "ecos2_matches.csv"))